In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

#Create interactive map
import folium
from folium.plugins import FastMarkerCluster

# Ignore warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

# Add project root to sys.path
root_path = Path.cwd().parent 
if str(root_path) not in sys.path:
    sys.path.insert(0, str(root_path))

#Import
from pipelines.data_pipeline import load_raw_data, clean_data, accident_engineer_features, complaints_engineer_features, create_temporal_features
from pipelines.data_pipeline import generate_hourly_heatmap, generate_accident_map # functions to create maps


Libraries imported successfully!


In [2]:
#Load the City Traffic Accident Database
df_City_Traffic = load_raw_data("city_traffic_accidents.csv")
missing_count = df_City_Traffic.isnull().sum()

print(missing_count)

ID                            0
Source                        0
Severity                      0
Start_Time                    0
End_Time                      0
Start_Lat                     0
Start_Lng                     0
End_Lat                  219978
End_Lng                  219978
Distance(mi)                  0
Description                   0
Street                      654
City                         28
County                        0
State                         0
Zipcode                     118
Country                       0
Timezone                    476
Airport_Code               1514
Weather_Timestamp          7842
Temperature(F)            10700
Wind_Chill(F)            129188
Humidity(%)               11371
Pressure(in)               9170
Visibility(mi)            11482
Wind_Direction            11382
Wind_Speed(mph)           36902
Precipitation(in)        142366
Weather_Condition         11300
Amenity                       0
Bump                          0
Crossing

In [3]:

#df_Complaints = load_raw_data("urbanpulse_311_complaints.csv")
df_City_Traffic['Start_Time'] = pd.to_datetime(df_City_Traffic['Start_Time'], errors='coerce')
df_City_Traffic['End_Time'] = pd.to_datetime(df_City_Traffic['End_Time'], errors='coerce')

#Clean data based on common cleaning features
df_City_Traffic = clean_data(df_City_Traffic)
#df_Complaints = clean_data(df_Complaints)

#Clean data and engineer features for accidents (includes temporal features)
df_City_Traffic = accident_engineer_features(df_City_Traffic)
#df_Complaints = complaints_engineer_features(df_Complaints)

# Verify it worked
print(f"Total records loaded: {len(df_City_Traffic)}")
#print(f"Total records loaded: {len(df_Complaints)}")
print(df_City_Traffic.head())

  Calculating sun data for 1561 rows...
  Filling weather with regional medians...
Total records loaded: 500000
   Severity          Start_Time            End_Time  Start_Lat   Start_Lng  \
0         2 2019-10-29 13:16:54 2019-10-29 15:21:34  35.834797  -78.638512   
1         2 2021-10-13 06:30:00 2021-10-13 06:59:15  36.088970  -96.011734   
2         2 2022-08-14 14:42:58 2022-08-14 16:27:58  33.537049  -86.794445   
3         2 2021-06-25 19:13:44 2021-06-25 20:42:30  34.071722 -117.612886   
4         2 2022-03-18 12:50:30 2022-03-18 13:13:00  40.324235  -76.790464   

     End_Lat     End_Lng  Distance(mi)        City          County State  \
0  35.834797  -78.638512         0.000     raleigh            wake    nc   
1  36.088970  -96.011734         0.000       tulsa           tulsa    ok   
2  33.535373  -86.796156         0.152  birmingham       jefferson    al   
3  34.078917 -117.625339         0.869     ontario  san bernardino    ca   
4  40.322625  -76.788114         0.166 

In [4]:
df_City_Traffic.head()
# 1. Use df.select_dtypes(include=[np.number]) to get numerical columns
numerical_df = df_City_Traffic.select_dtypes(include=[np.number])

# 2. Get the column names as a list with .columns.tolist()
numerical_cols= numerical_df.columns.tolist()

# 4. Print the count and list of numerical features
print(f'Count: {len(numerical_cols)}')
print(f'Features: {numerical_cols}')


Count: 59
Features: ['Severity', 'Start_Lat', 'Start_Lng', 'End_Lat', 'End_Lng', 'Distance(mi)', 'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)', 'Geo_Cluster', 'dist_from_reg_hotspot', 'hour', 'day_of_week', 'month', 'is_weekend', 'is_morning_rush', 'is_evening_rush', 'is_rush_hour', 'duration_min', 'weather_data_available', 'is_freezing', 'low_visibility_severity', 'has_precipitation', 'weather_cluster_clear', 'weather_cluster_cloudy', 'weather_cluster_low_visibility', 'weather_cluster_other', 'weather_cluster_rain', 'weather_cluster_snow_ice', 'weather_cluster_storm', 'region_Midwest', 'region_Northeast', 'region_Other', 'region_South', 'region_West', 'word_accident', 'word_exit', 'word_blocked', 'word_incident', 'word_lane', 'word_traffic', 'word_caution', 'word_drive', 'word_slow', 'word_closed', 'word_right', 'word_northbound', 'word_southbound', 'word_stationary', 'word_eastbound', 'word_westbound', 'word

In [5]:
# if len(numerical_cols) > 0:
#     n_cols = 3
#     n_rows = (len(numerical_cols) + n_cols - 1) // n_cols

#     fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
#     axes = axes.flatten() if n_rows > 1 else [axes] if n_rows == 1 and n_cols == 1 else axes

#     for i, col in enumerate(numerical_cols):
#         axes[i].hist(df_City_Traffic[col].dropna(), bins=30, edgecolor='black', alpha=0.7)
#         axes[i].set_title(col)
#         axes[i].set_xlabel('')

#     # Hide empty subplots
#     for j in range(len(numerical_cols), len(axes)):
#         axes[j].set_visible(False)

#     plt.tight_layout()
#     plt.show()
# else:
#     print("No numerical features found (besides target).")

While our primary client was Nova Haven, we trained our model on national-scale data to ensure the AI understood broader traffic disruption patterns before fine-tuning it for local city operations.

In [6]:
df_City_Traffic.head()

,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),City,County,State,Zipcode,Timezone,Airport_Code,Weather_Timestamp,Temperature(F),Wind_Chill(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Direction,Wind_Speed(mph),Precipitation(in),Amenity,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight,Geo_Cluster,dist_from_reg_hotspot,hour,day_of_week,month,is_weekend,is_morning_rush,is_evening_rush,is_rush_hour,duration_min,weather_data_available,is_freezing,low_visibility_severity,has_precipitation,weather_cluster_clear,weather_cluster_cloudy,weather_cluster_low_visibility,weather_cluster_other,weather_cluster_rain,weather_cluster_snow_ice,weather_cluster_storm,region_Midwest,region_Northeast,region_Other,region_South,region_West,word_accident,word_exit,word_blocked,word_incident,word_lane,word_traffic,word_caution,word_drive,word_slow,word_closed,word_right,word_northbound,word_southbound,word_stationary,word_eastbound,word_westbound,word_shoulder,word_left,word_crash,word_delays
0,2,2019-10-29 13:16:54,2019-10-29 15:21:34,35.834797,-78.638512,35.834797,-78.638512,0.000,raleigh,wake,nc,27609,us/eastern,krdu,2019-10-29 12:51:00,69.0,69.0,73.0,29.77,10.0,e,5.0,0.00,False,False,False,False,False,False,False,False,False,False,False,False,False,day,day,day,day,3,277.840073,13.0,1.0,10.0,0,0,0,0,124.666667,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2,2021-10-13 06:30:00,2021-10-13 06:59:15,36.088970,-96.011734,36.088970,-96.011734,0.000,tulsa,tulsa,ok,74107,us/central,krvs,2021-10-13 06:29:00,76.0,76.0,85.0,29.01,10.0,s,8.0,0.01,False,False,False,False,False,False,False,False,False,False,False,False,False,night,night,night,day,7,310.642362,6.0,2.0,10.0,0,0,0,0,29.250000,1,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
2,2,2022-08-14 14:42:58,2022-08-14 16:27:58,33.537049,-86.794445,33.535373,-86.796156,0.152,birmingham,jefferson,al,35234,us/central,kbhm,2022-08-14 14:53:00,89.0,89.0,46.0,29.33,10.0,var,3.0,0.00,False,False,False,False,True,False,False,False,False,False,False,False,False,day,day,day,day,3,307.199394,14.0,6.0,8.0,1,0,0,0,105.000000,1,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
3,2,2021-06-25 19:13:44,2021-06-25 20:42:30,34.071722,-117.612886,34.078917,-117.625339,0.869,ontario,san bernardino,ca,91764,us/pacific,kont,2021-06-25 18:53:00,84.0,84.0,32.0,28.84,10.0,w,9.0,0.00,False,False,False,False,False,False,False,False,False,False,False,False,False,day,day,day,day,5,39.000189,19.0,4.0,6.0,0,0,1,1,88.766667,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0
4,2,2022-03-18 12:50:30,2022-03-18 13:13:00,40.324235,-76.790464,40.322625,-76.788114,0.166,harrisburg,dauphin,pa,17112,us/eastern,kcxy,2022-03-18 12:56:00,71.0,71.0,47.0,29.63,10.0,ene,5.0,0.00,False,False,False,False,False,False,False,False,False,False,False,False,False,day,day,day,day,0,87.076755,12.0,4.0,3.0,0,0,0,0,22.500000,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0


In [7]:
missing_count = df_City_Traffic.isnull().sum()

print(missing_count)

Severity                               0
Start_Time                           892
End_Time                             892
Start_Lat                              0
Start_Lng                              0
End_Lat                                0
End_Lng                                0
Distance(mi)                           0
City                                   0
County                                 0
State                                  0
Zipcode                                0
Timezone                               2
Airport_Code                        1514
Weather_Timestamp                    892
Temperature(F)                         0
Wind_Chill(F)                     129188
Humidity(%)                            0
Pressure(in)                           0
Visibility(mi)                         0
Wind_Direction                         0
Wind_Speed(mph)                        0
Precipitation(in)                      0
Amenity                                0
Bump            

In [8]:
df_City_Traffic = df_City_Traffic.dropna(axis=1) 
missing_count = df_City_Traffic.isnull().sum()

print(missing_count)

Severity                          0
Start_Lat                         0
Start_Lng                         0
End_Lat                           0
End_Lng                           0
Distance(mi)                      0
City                              0
County                            0
State                             0
Zipcode                           0
Temperature(F)                    0
Humidity(%)                       0
Pressure(in)                      0
Visibility(mi)                    0
Wind_Direction                    0
Wind_Speed(mph)                   0
Precipitation(in)                 0
Amenity                           0
Bump                              0
Crossing                          0
Give_Way                          0
Junction                          0
No_Exit                           0
Railway                           0
Roundabout                        0
Station                           0
Stop                              0
Traffic_Calming             

In [9]:
df_City_Traffic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 72 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   Severity                        500000 non-null  int64  
 1   Start_Lat                       500000 non-null  float64
 2   Start_Lng                       500000 non-null  float64
 3   End_Lat                         500000 non-null  float64
 4   End_Lng                         500000 non-null  float64
 5   Distance(mi)                    500000 non-null  float64
 6   City                            500000 non-null  object 
 7   County                          500000 non-null  object 
 8   State                           500000 non-null  object 
 9   Zipcode                         500000 non-null  object 
 10  Temperature(F)                  500000 non-null  float64
 11  Humidity(%)                     500000 non-null  float64
 12  Pressure(in)    

In [10]:
df_City_Traffic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 72 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   Severity                        500000 non-null  int64  
 1   Start_Lat                       500000 non-null  float64
 2   Start_Lng                       500000 non-null  float64
 3   End_Lat                         500000 non-null  float64
 4   End_Lng                         500000 non-null  float64
 5   Distance(mi)                    500000 non-null  float64
 6   City                            500000 non-null  object 
 7   County                          500000 non-null  object 
 8   State                           500000 non-null  object 
 9   Zipcode                         500000 non-null  object 
 10  Temperature(F)                  500000 non-null  float64
 11  Humidity(%)                     500000 non-null  float64
 12  Pressure(in)    

In [11]:
df_City_Traffic['Wind_Direction'].unique()

array(['e', 's', 'var', 'w', 'ene', 'nne', 'calm', 'ssw', 'sse', 'nnw',
       'ese', 'se', 'n', 'wsw', 'sw', 'north', 'nw', 'ne', 'wnw', 'east',
       'variable', 'west', 'south'], dtype=object)